<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_08_model_tuning/stage_08_03_gru.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_08_03 - Tuning - GRU**

**GRU**

- Tuneo grueso

  * `hidden_size` → capacidad del modelo

    * `[64, 128, 256]`
  * `num_layers`

    * `[1, 2]`
  * `learning_rate`

    * `[1e-4, 5e-4, 1e-3]`

- Tuneo fino

  * `dropout` → `[0.0, 0.1, 0.2, 0.3]`
  * `batch_size` → `[1024, 2048, 4096]`
  * `grad_clip_norm` → `[0.5, 1.0, 2.0]`
  * thresholds de probabilidad

# **SETUP COMÚN DEL PIPELINE DE ENTRENAMIENTO**

## **1. Imports**

In [1]:
# Permite anotaciones modernas en Python < 3.11
from __future__ import annotations

# ================================
# Standard library
# ================================
import os
import sys
import json
import time
import random
import importlib
import warnings
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Iterable, Tuple

# ================================
# Third-party
# ================================
import numpy as np
import joblib

# ================================
# Configuración global
# ================================

# ---- Warnings (controlado, no agresivo)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Logging limpio para notebooks
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.propagate = False  # evita duplicación con el root logger

# limpiar handlers si se re-ejecuta la celda
if logger.handlers:
    logger.handlers.clear()

handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.info("Environment initialized")

2026-04-23 16:22:36,738 | INFO | Environment initialized


## **2. Acceso a drive**

In [2]:
# ================================
# Entorno (Google Drive / local)
# ================================

from pathlib import Path
import os

# Detectar si estamos en Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DEFAULT_DRIVE_DIR = "/content/drive/MyDrive/neural_profit/"
else:
    # fallback local (puedes ajustarlo si quieres)
    DEFAULT_DRIVE_DIR = "./neural_profit/"

# Ruta base del proyecto
DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", DEFAULT_DRIVE_DIR))

# Validación básica
if not DRIVE_DIR.exists():
    logger.warning(f"DRIVE_DIR no existe: {DRIVE_DIR}")
else:
    logger.info(f"DRIVE_DIR: {DRIVE_DIR}")

Mounted at /content/drive


2026-04-23 16:23:10,356 | INFO | DRIVE_DIR: /content/drive/MyDrive/neural_profit


## **3. Rutas de ventanas `seq2one` y `scaler`**

In [3]:
# ================================
# Configuración de paths
# ================================

WINDOWS_SEQ2ONE_DIR = DRIVE_DIR / os.environ.get(
    "WINDOWS_SEQ2ONE_DIR",
    "data/07_windows/seq2one/"
)

SCALERS_DIR = DRIVE_DIR / os.environ.get(
    "SCALERS_DIR",
    "data/06_scaled/"
)

# Validación básica
for p in [WINDOWS_SEQ2ONE_DIR, SCALERS_DIR]:
    if not p.exists():
        logger.warning(f"Path no existe: {p}")
    else:
        logger.info(f"Path OK: {p}")

# ================================
# Configuración del experimento
# ================================

# Targets T2 (clasificación)
TARGETS = [
  "t2_p40_h30",
  "t2_p40_h60",
  "t2_p50_h30",
]

# Tamaños de ventana
WINDOW_SIZES = [30]

# Splits
SPLITS = ["train", "valid", "test"]

# Features finales (consistencia global)
FEATURES_T2 = [
        "ema_60",
        "roc_60",
        "roc_30",
        "stoch_k_30",
        "mom_5",
        "atr_norm_10",
        "macd"
]

logger.info("Configuración de experimento cargada")
logger.info(f"Targets: {TARGETS}")
logger.info(f"Window sizes: {WINDOW_SIZES}")

2026-04-23 16:23:11,643 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/07_windows/seq2one
2026-04-23 16:23:11,644 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/06_scaled
2026-04-23 16:23:11,644 | INFO | Configuración de experimento cargada
2026-04-23 16:23:11,645 | INFO | Targets: ['t2_p40_h30', 't2_p40_h60', 't2_p50_h30']
2026-04-23 16:23:11,646 | INFO | Window sizes: [30]


In [4]:
# ================================
# Construcción y validación de paths (windows + scaler compartido)
# ================================

def build_windows_paths(window_sizes, targets, splits, base_dir):
    """
    windows_paths[w][t][s] -> Path
    """
    windows_paths = {}

    for w in window_sizes:
        windows_paths[w] = {}
        for t in targets:
            windows_paths[w][t] = {}
            for s in splits:
                windows_paths[w][t][s] = (
                    base_dir
                    / f"L{w}"
                    / f"windows_{t}_{s}.npz"
                )

    return windows_paths


def build_shared_scaler_path(base_dir, scaler_name="scaler_t2.pkl"):
    """
    Retorna el path del scaler compartido para T2.
    """
    return base_dir / scaler_name


# ================================
# Construcción
# ================================

WINDOWS_PATHS = build_windows_paths(
    window_sizes=WINDOW_SIZES,
    targets=TARGETS,
    splits=SPLITS,
    base_dir=WINDOWS_SEQ2ONE_DIR,
)

SCALER_T2_PATH = build_shared_scaler_path(
    base_dir=SCALERS_DIR,
    scaler_name="scaler_mnq_t2.pkl",
)

logger.info("Paths construidos (windows + scaler compartido T2)")

# ================================
# Validación
# ================================

missing_windows = []
existing_windows = []

for w in WINDOW_SIZES:
    for t in TARGETS:
        for s in SPLITS:
            p = WINDOWS_PATHS[w][t][s]
            if p.exists():
                existing_windows.append(p)
            else:
                missing_windows.append(p)

scaler_exists = SCALER_T2_PATH.exists()

# -------------------------------
# Logs
# -------------------------------
logger.info(f"Windows OK      : {len(existing_windows)}")
logger.info(f"Windows missing : {len(missing_windows)}")

if scaler_exists:
    logger.info(f"Scaler T2 OK    : {SCALER_T2_PATH}")
else:
    logger.warning(f"Scaler T2 missing: {SCALER_T2_PATH}")

if missing_windows:
    logger.warning("Archivos de ventanas faltantes:")
    for p in missing_windows[:10]:
        logger.warning(f"Missing window: {p}")

if not missing_windows and scaler_exists:
    logger.info("Todos los archivos existen correctamente")

# ================================
# Ejemplo de uso
# ================================

# window
# WINDOWS_PATHS[30]["t2_p40_h30"]["train"]

# scaler compartido
# SCALER_T2_PATH

2026-04-23 16:23:11,655 | INFO | Paths construidos (windows + scaler compartido T2)
2026-04-23 16:23:12,528 | INFO | Windows OK      : 9
2026-04-23 16:23:12,529 | INFO | Windows missing : 0
2026-04-23 16:23:12,530 | INFO | Scaler T2 OK    : /content/drive/MyDrive/neural_profit/data/06_scaled/scaler_mnq_t2.pkl
2026-04-23 16:23:12,530 | INFO | Todos los archivos existen correctamente


## **4. Carga de ventanas X/y**

### **4.1. Cargar ventanas (*.npz)**

In [5]:
from pathlib import Path
from typing import Tuple
import numpy as np


def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar (T2 seq2one).

    Espera:
    - 'X': (n_samples, seq_len, n_features)
    - 'y' o 'Y': (n_samples,) o (n_samples, 1) o (n_samples, seq_len, 1)

    Retorna
    -------
    X : np.ndarray  -> (n_samples, seq_len, n_features)
    y : np.ndarray  -> (n_samples,)
    """

    # --------------------------------------------------
    # 1. Validación
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")

        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # copiar a memoria
        X = X.copy()
        y = y.copy()

    # --------------------------------------------------
    # 3. Ajuste de shape de y (CRÍTICO)
    # --------------------------------------------------
    # Caso (n_samples, 1)
    if y.ndim == 2 and y.shape[1] == 1:
        y = y.reshape(-1)

    # Caso (n_samples, seq_len, 1) -> tomar último valor
    elif y.ndim == 3:
        y = y[:, -1, 0]

    # Caso (n_samples, seq_len) -> tomar último valor
    elif y.ndim == 2 and y.shape[1] > 1:
        y = y[:, -1]

    # Validación final
    if y.ndim != 1:
        raise ValueError(f"y no es 1D después de procesamiento: shape={y.shape}")

    # --------------------------------------------------
    # 4. Logs útiles
    # --------------------------------------------------
    logger.info(f"Loaded: {path.name}")
    logger.info(f"X shape: {X.shape} | y shape: {y.shape}")

    return X, y

### **4.2. Cargar escalador (*.pkl)**

In [6]:
# --------------------------------------------------
# Función común: carga scaler compartido (T2)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib.

    Parámetros
    ----------
    path : Path
        Ruta al archivo scaler (.pkl)

    Retorna
    -------
    scaler : Any
        Objeto scaler (ej. StandardScaler, MinMaxScaler)
    """

    # --------------------------------------------------
    # 1. Validación de existencia
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    scaler = joblib.load(path)

    # --------------------------------------------------
    # 3. Validación básica (opcional pero útil)
    # --------------------------------------------------
    if not hasattr(scaler, "transform"):
        raise TypeError(f"El objeto cargado no es un scaler válido: {type(scaler)}")

    # --------------------------------------------------
    # 4. Logging
    # --------------------------------------------------
    logger.info(f"Scaler cargado: {path.name}")

    return scaler

### **4.3. Función de carga**

In [7]:
from typing import Any, Dict, Mapping
from pathlib import Path


# --------------------------------------------------
# Carga completa: ventanas + scaler compartido T2
# --------------------------------------------------
def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scaler_path: Path,
) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler compartido T2
    para un par (window_size, target).

    Parámetros
    ----------
    window_size : int
        Tamaño de ventana.
    target : str
        Target T2, por ejemplo:
        - 't2_p40_h30'
        - 't2_p40_h60'
        - 't2_p50_h30'
    windows_paths : mapping
        Estructura:
        windows_paths[window_size][target][split] -> Path
    scaler_path : Path
        Ruta al scaler compartido T2.

    Retorna
    -------
    bundle : dict
        Diccionario con:
        - metadata
        - paths
        - scaler
        - train/valid/test con X e y
    """

    # --------------------------
    # 1) Validaciones
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    for split in ["train", "valid", "test"]:
        if split not in windows_paths[window_size][target]:
            raise KeyError(
                f"split='{split}' no existe en windows_paths[{window_size}]['{target}']"
            )

    if not scaler_path.exists():
        raise FileNotFoundError(f"No se encontró el scaler compartido: {scaler_path}")

    # --------------------------
    # 2) Paths
    # --------------------------
    train_path = windows_paths[window_size][target]["train"]
    valid_path = windows_paths[window_size][target]["valid"]
    test_path  = windows_paths[window_size][target]["test"]

    # --------------------------
    # 3) Carga de ventanas
    # --------------------------
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test, y_test   = load_npz_windows(test_path)

    # --------------------------
    # 4) Carga de scaler
    # --------------------------
    scaler = load_scaler(scaler_path)

    # --------------------------
    # 5) Inferir horizonte
    # --------------------------
    try:
        horizon_str = target.split("_")[-1]   # ej: "h30"
        horizon = int(horizon_str.replace("h", ""))
    except Exception:
        horizon = None

    # --------------------------
    # 6) Logging
    # --------------------------
    logger.info(
        f"Bundle cargado | target={target} | window_size={window_size} | "
        f"train={X_train.shape} | valid={X_valid.shape} | test={X_test.shape}"
    )

    # --------------------------
    # 7) Retorno
    # --------------------------
    return {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {
            "X": X_train,
            "y": y_train,
        },
        "valid": {
            "X": X_valid,
            "y": y_valid,
        },
        "test": {
            "X": X_test,
            "y": y_test,
        },
    }

### **4.4. Creación de bundles T2**

In [8]:
def create_bundles(
    window_size: int,
    targets: list[str] = TARGETS,
    windows_paths=WINDOWS_PATHS,
    scaler_path=SCALER_T2_PATH,
):
    """
    Crea un bundle por target para un window_size dado.

    Retorna
    -------
    bundles : dict
        bundles[target] -> bundle dict
    """

    bundles = {}

    # --------------------------
    # Construcción
    # --------------------------
    for target in targets:
        bundles[target] = load_windows_and_scaler(
            window_size=window_size,
            target=target,
            windows_paths=windows_paths,
            scaler_path=scaler_path,
        )

    # --------------------------
    # Verificación rápida
    # --------------------------
    print(f"\n{'=' * 70}")
    print(f"WINDOW_SIZE: {window_size}")
    print(f"{'=' * 70}")

    for target in targets:
        b = bundles[target]

        print(f"\nTARGET: {target}")
        print("Train :", b["train"]["X"].shape, b["train"]["y"].shape)
        print("Valid :", b["valid"]["X"].shape, b["valid"]["y"].shape)
        print("Test  :", b["test"]["X"].shape,  b["test"]["y"].shape)
        print("Scaler:", type(b["scaler"]).__name__)

    return bundles

In [9]:
bundles_L30 = create_bundles(window_size=30)

2026-04-23 16:23:13,966 | INFO | Loaded: windows_t2_p40_h30_train.npz
2026-04-23 16:23:13,967 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-23 16:23:14,628 | INFO | Loaded: windows_t2_p40_h30_valid.npz
2026-04-23 16:23:14,629 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-23 16:23:15,536 | INFO | Loaded: windows_t2_p40_h30_test.npz
2026-04-23 16:23:15,536 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-23 16:23:17,659 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-23 16:23:17,659 | INFO | Bundle cargado | target=t2_p40_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)
2026-04-23 16:23:20,784 | INFO | Loaded: windows_t2_p40_h60_train.npz
2026-04-23 16:23:20,785 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-23 16:23:21,753 | INFO | Loaded: windows_t2_p40_h60_valid.npz
2026-04-23 16:23:21,754 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-23 16:23:22,565 | INFO | Loaded: windows_t2_p4


WINDOW_SIZE: 30

TARGET: t2_p40_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p40_h60
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p50_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler


Como acceder a las ventanas X e y:

```python
bundle_p40_h30 = bundles_L30["t2_p40_h30"]

X_train = bundle_p40_h30["train"]["X"]
y_train = bundle_p40_h30["train"]["y"]

X_valid = bundle_p40_h30["valid"]["X"]
y_valid = bundle_p40_h30["valid"]["y"]

X_test = bundle_p40_h30["test"]["X"]
y_test = bundle_p40_h30["test"]["y"]

scaler = bundle_p40_h30["scaler"]

print("Target :", bundle_p40_h30["target"])
print("Horizon:", bundle_p40_h30["horizon"])
print("Train  :", X_train.shape, y_train.shape)
print("Valid  :", X_valid.shape, y_valid.shape)
print("Test   :", X_test.shape, y_test.shape)
print("Scaler :", type(scaler).__name__)
```




In [10]:
bundles_L30

{'t2_p40_h30': {'window_size': 30,
  'target': 't2_p40_h30',
  'horizon': 30,
  'paths': {'train': '/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_p40_h30_train.npz',
   'valid': '/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_p40_h30_valid.npz',
   'test': '/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_p40_h30_test.npz',
   'scaler': '/content/drive/MyDrive/neural_profit/data/06_scaled/scaler_mnq_t2.pkl'},
  'scaler': StandardScaler(),
  'train': {'X': array([[[-0.11680385, -0.06583842, -0.2238865 , ..., -0.04129868,
            -1.507039  , -0.02778307],
           [ 0.0909589 ,  0.05818945, -0.07248875, ...,  0.3872164 ,
            -1.3723946 ,  0.08631181],
           [-0.18229802, -0.13053213, -0.21077001, ..., -0.1151728 ,
            -1.2194692 ,  0.02307655],
           ...,
           [ 0.40151003,  0.28442496,  0.59289074, ...,  0.47497907,
            -1.0690751 , -0.19294474],
         

### **4.5. Preparación de inputs según el tipo de modelo**

In [11]:
# ============================================================
# Preparación de inputs según el tipo de modelo
# ============================================================

def flatten_seq2one_X(X: np.ndarray) -> np.ndarray:
    """
    Aplana ventanas seq2one para modelos tabulares.

    Convierte:
        (n_samples, seq_len, n_features)
    a:
        (n_samples, seq_len * n_features)
    """
    X = np.asarray(X)

    if X.ndim != 3:
        raise ValueError(
            f"Se esperaba X 3D con shape (n, seq_len, n_features). "
            f"Recibido X.shape={X.shape}"
        )

    n_samples = X.shape[0]
    return X.reshape(n_samples, -1)


def prepare_X_for_model(
    X: np.ndarray,
    *,
    input_mode: str,
) -> np.ndarray:
    """
    Prepara X según el tipo de modelo.

    Parámetros
    ----------
    X : np.ndarray
        Array de entrada.
    input_mode : str
        - '2d_flat' : aplana ventanas 3D a 2D
        - '3d'      : deja X como está

    Retorna
    -------
    np.ndarray
        X transformado para el modelo correspondiente.
    """
    X = np.asarray(X)

    if input_mode == "3d":
        if X.ndim != 3:
            raise ValueError(
                f"Se esperaba X 3D para input_mode='3d'. "
                f"Recibido X.shape={X.shape}"
            )
        return X

    if input_mode == "2d_flat":
        return flatten_seq2one_X(X)

    raise ValueError(
        f"input_mode no soportado: {input_mode}. "
        f"Use '2d_flat' o '3d'."
    )

**Cómo se usa después**

Para Logistic Regression:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="2d_flat")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="2d_flat")
```

Para LSTM / GRU / Transformer:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="3d")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="3d")
```


## **5. Sanity Check**

Estos sanity checks sirven para verificar, antes de entrenar, que los datos cargados tengan la estructura correcta y no vengan con errores silenciosos.

En concreto, comprueban que:

- X tenga el formato esperado: 3D (n, seq_len, n_features) o 2D (n, d_flat)
- y tenga forma válida para clasificación seq2one
- X e y tengan la misma cantidad de muestras
- no haya NaN ni inf
- las dimensiones sean consistentes entre train, valid y test
- exista más de una clase en y

Nos conviene tenerlos, porque ayudan a detectar errores de shape o de datos antes de llegar al entrenamiento.

In [12]:
from __future__ import annotations

from typing import Any, Optional, Tuple, Dict, Mapping
import numpy as np


# ============================================================
# SANITY CHECKS PARA DATASETS SEQ2ONE (T2)
# ============================================================

def _as_numpy_array(a: Any, *, name: str) -> np.ndarray:
    """
    Convierte una entrada a np.ndarray y valida que no esté vacía
    ni contenga NaN/inf.
    """
    arr = np.asarray(a)

    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")

    if not np.isfinite(arr).all():
        raise ValueError(
            f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}"
        )

    return arr


def _normalize_y_seq2one(
    y: np.ndarray,
    *,
    name: str = "y",
    allow_seq_inputs_take_last: bool = False,
) -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).

    Acepta:
    - (n,)
    - (n, 1)
    - (n, seq_len)       si allow_seq_inputs_take_last=True
    - (n, seq_len, 1)    si allow_seq_inputs_take_last=True
    """
    if y.ndim == 1:
        return y

    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)

    if allow_seq_inputs_take_last:
        if y.ndim == 2 and y.shape[1] > 1:
            return y[:, -1]

        if y.ndim == 3 and y.shape[2] == 1:
            return y[:, -1, 0]

    raise ValueError(
        f"{name} shape inválido para seq2one. "
        f"Se esperaba (n,) o (n,1)"
        f"{' o secuencial si allow_seq_inputs_take_last=True' if allow_seq_inputs_take_last else ''}. "
        f"Recibido {y.shape}"
    )


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiere (seq_len, n_features, mode) desde X.

    mode:
    - '3d': X = (n, seq_len, n_features)
    - '2d': X = (n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"

    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"

    raise ValueError(
        f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}"
    )


def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one de clasificación.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)
      - y secuencial, opcionalmente, tomando el último valor

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: no aplica directamente.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D.
    allow_seq_inputs_take_last:
        Si y viene como secuencia, toma el último valor.
    """
    X = _as_numpy_array(X, name=f"X[{split_name}]")
    y = _as_numpy_array(y, name=f"y[{split_name}]")

    y = _normalize_y_seq2one(
        y,
        name=f"y[{split_name}]",
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
    )

    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: "
            f"X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    if mode == "3d":
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, "
                f"recibido={seq_len}. X.shape={X.shape}"
            )

        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, "
                f"recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, "
                f"recibido={d_flat}. X.shape={X.shape}"
            )

        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim o pase X en 3D."
            )

    classes, counts = np.unique(y, return_counts=True)
    class_distribution = {
        str(cls): int(cnt) for cls, cnt in zip(classes, counts)
    }

    if len(classes) < 2:
        raise ValueError(
            f"{split_name}: y contiene menos de 2 clases únicas. "
            f"classes={classes.tolist()}"
        )

    info = {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "n_classes": int(len(classes)),
        "classes": classes.tolist(),
        "class_distribution": class_distribution,
    }

    if verbose:
        print(
            f"[sanity_check_seq2one] {split_name} | "
            f"X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | n_classes={info['n_classes']} | "
            f"classes={info['classes']}"
        )

    return info


def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "target": "t2_p40_h30",
      "horizon": 30,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Si no se pasan expected_*, usa TRAIN como referencia.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        expected_n_features = None

    out_tr = sanity_check_seq2one(
        X_tr,
        y_tr,
        f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    out_va = sanity_check_seq2one(
        X_va,
        y_va,
        f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    out_te = sanity_check_seq2one(
        X_te,
        y_te,
        f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    target = bundle.get("target", "NA")
    horizon = bundle.get("horizon", "NA")

    if verbose:
        print(f"OK {tag} | target={target} | horizon={horizon}")

    return {
        "target": target,
        "horizon": horizon,
        "train": out_tr,
        "valid": out_va,
        "test": out_te,
    }


def run_sanity_checks_all_bundles_seq2one(
    bundles: Mapping[str, Dict[str, Any]],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Corre sanity checks para todos los bundles de un diccionario:

    bundles[target] -> bundle
    """
    results = {}

    for target, bundle in bundles.items():
        results[target] = run_sanity_checks_for_bundle_seq2one(
            bundle,
            tag=target,
            verbose=verbose,
        )

    return results

In [13]:
sanity_results = run_sanity_checks_all_bundles_seq2one(
    bundles_L30,
    verbose=True,
)

[sanity_check_seq2one] train_t2_p40_h30 | X=(32147, 30, 7) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p40_h30 | X=(6882, 30, 7) | y=(6882,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] test_t2_p40_h30 | X=(6913, 30, 7) | y=(6913,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
OK t2_p40_h30 | target=t2_p40_h30 | horizon=30
[sanity_check_seq2one] train_t2_p40_h60 | X=(32147, 30, 7) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p40_h60 | X=(6882, 30, 7) | y=(6882,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] test_t2_p40_h60 | X=(6913, 30, 7) | y=(6913,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
OK t2_p40_h60 | target=t2_p40_h60 | horizon=60
[sanity_check_seq2one] train_t2_p50_h30 | X=(32147, 30, 7) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p50_h30 | X=(6882, 30, 7) | y=(6882,) | mode=3d | n_classes=3 | c

## **6. Módulo de métricas T2**

In [14]:
# ================================
# Setup para importar módulos del proyecto
# ================================

import sys
import importlib

if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))

# asegurar que metrics es paquete
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

importlib.invalidate_caches()

# ================================
# Imports de métricas
# ================================

from metrics.classification_metrics import (
    compute_classification_metrics,
    metrics_to_flat_dict,
    print_classification_report_block,
)

from metrics.classification_probabilities import (
    compute_probabilistic_outputs,
    apply_decision_rule,
)

print("Módulos importados correctamente")

Módulos importados correctamente


In [15]:
# ================================
# Utilidades: outputs -> DataFrame
# ================================

from __future__ import annotations

import pandas as pd
from typing import Any


def classification_metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    window_size: int,
    target: str,
    horizon: int | None = None,
) -> pd.DataFrame:
    """
    Convierte el output de compute_classification_metrics(...)
    en una fila de DataFrame.

    Usa metrics_to_flat_dict(...) para aplanar la salida
    del módulo classification_metrics y luego agrega metadata
    del experimento.
    """
    flat_metrics = metrics_to_flat_dict(metrics)

    row = {
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        **flat_metrics,
    }

    return pd.DataFrame([row])


def _flatten_dict(
    d: dict[str, Any],
    *,
    parent_key: str = "",
    sep: str = "_",
) -> dict[str, Any]:
    """
    Aplana un diccionario arbitrario de forma recursiva.

    Ejemplo:
    {"a": {"b": 1}} -> {"a_b": 1}
    """
    items: dict[str, Any] = {}

    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else str(k)

        if isinstance(v, dict):
            items.update(_flatten_dict(v, parent_key=new_key, sep=sep))
        else:
            items[new_key] = v

    return items


def probabilities_metrics_to_df(
    prob_metrics: dict,
    *,
    model: str,
    split: str,
    window_size: int,
    target: str,
    horizon: int | None = None,
) -> pd.DataFrame:
    """
    Convierte el output del módulo classification_probabilities
    en una fila de DataFrame.

    Como la estructura puede variar según la implementación,
    se aplana recursivamente el diccionario y luego se agrega
    metadata del experimento.
    """
    flat_prob_metrics = _flatten_dict(prob_metrics)

    row = {
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        **flat_prob_metrics,
    }

    return pd.DataFrame([row])


logger.info("Utilidades de exportación a DataFrame cargadas")

2026-04-23 16:23:30,530 | INFO | Utilidades de exportación a DataFrame cargadas


## **7. Persistencia de métricas (tracking de experimentos)**

In [16]:
# ================================
# Persistencia de métricas de clasificación
# ================================

from pathlib import Path
import pandas as pd


def load_classification_metrics_if_exists(
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics_tuning" / "classification_metrics",
) -> pd.DataFrame:
    """
    Carga métricas de clasificación si el archivo existe.

    El nombre del archivo incluye modelo y split para evitar
    mezclar resultados de distintos experimentos.
    """
    path = base_dir / f"classification_metrics_{model_name}_{split}.parquet"

    if path.exists():
        logger.info(f"Cargando métricas desde: {path}")
        return pd.read_parquet(path)

    logger.info(
        f"No existen métricas previas para model={model_name} | split={split}"
    )
    return pd.DataFrame()


def save_classification_metrics(
    df_metrics: pd.DataFrame,
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics_tuning" / "classification_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas de clasificación en formato Parquet.
    """
    base_dir.mkdir(parents=True, exist_ok=True)

    out_path = base_dir / f"classification_metrics_{model_name}_{split}.parquet"
    df_metrics.to_parquet(out_path, index=False)

    logger.info(f"Métricas guardadas en: {out_path}")

    return out_path


# ================================
# Persistencia de probabilidades / decisión
# ================================

def load_classification_probabilities_if_exists(
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics_tuning" / "classification_probabilities",
) -> pd.DataFrame:
    """
    Carga resultados probabilísticos si el archivo existe.

    El nombre del archivo incluye modelo y split para evitar
    mezclar resultados de distintos experimentos.
    """
    path = base_dir / f"classification_probabilities_{model_name}_{split}.parquet"

    if path.exists():
        logger.info(f"Cargando probabilidades desde: {path}")
        return pd.read_parquet(path)

    logger.info(
        f"No existen probabilidades previas para model={model_name} | split={split}"
    )
    return pd.DataFrame()


def save_classification_probabilities(
    df_probabilities: pd.DataFrame,
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics_tuning" / "classification_probabilities",
) -> Path:
    """
    Guarda un DataFrame de probabilidades / decisión en formato Parquet.
    """
    base_dir.mkdir(parents=True, exist_ok=True)

    out_path = base_dir / f"classification_probabilities_{model_name}_{split}.parquet"
    df_probabilities.to_parquet(out_path, index=False)

    logger.info(f"Probabilidades guardadas en: {out_path}")

    return out_path

Ejemplo de uso con logistic regression:

```python
df_metrics_all = load_classification_metrics_if_exists(
    model_name="logistic_regression",
    split="valid",
)

df_probabilities_all = load_classification_probabilities_if_exists(
    model_name="logistic_regression",
    split="valid",
)
```

Guardar:
```python
save_classification_metrics(
    df_metrics_all,
    model_name="logistic_regression",
    split="valid",
)

save_classification_probabilities(
    df_probabilities_all,
    model_name="logistic_regression",
    split="valid",
)
```

## **8. Gestión de dispositivo y memoria**

In [17]:
# ================================
# Device y limpieza de memoria
# ================================

def get_torch_device():
    """
    Retorna el device para modelos PyTorch.
    """
    import torch

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Using device: {device}")
    return device


def clear_torch_memory() -> None:
    """
    Limpia memoria Python y, si existe CUDA, libera caché GPU.
    Útil entre entrenamientos de modelos PyTorch.
    """
    import gc

    gc.collect()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass

## **9. Reproducibilidad**

In [18]:
# ================================
# Reproducibilidad
# ================================

def set_seeds(seed: int = 42) -> None:
    """
    Fija semillas para reproducibilidad en:
    - Python
    - NumPy
    - PyTorch (si está disponible)
    """

    import random
    import numpy as np
    import os

    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    # ---- PyTorch (si está instalado)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        # determinismo (más lento pero reproducible)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    except ImportError:
        pass  # no hay PyTorch, ignorar

    logger.info(f"Seeds fijadas en {seed}")


# Aplicación
SEED = 42
set_seeds(SEED)

2026-04-23 16:23:35,009 | INFO | Seeds fijadas en 42


# **10. Entrenamiento de modelo**

## **10.1. Función unitaria por bundle**

In [19]:
import copy
import numpy as np
import torch
import torch.nn as nn


class GRUClassifier(nn.Module):
    def __init__(
        self,
        n_features: int,
        hidden_size: int = 64,
        num_layers: int = 1,
        dropout: float = 0.0,
        num_classes: int = 3,
    ):
        super().__init__()

        gru_dropout = dropout if num_layers > 1 else 0.0

        self.gru = nn.GRU(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=gru_dropout,
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.gru(x)          # (batch, seq_len, hidden_size)
        last_out = out[:, -1, :]      # many-to-one
        last_out = self.dropout(last_out)
        logits = self.fc(last_out)    # (batch, num_classes)
        return logits


def run_gru_for_bundle_seq2one(
    bundle,
    *,
    hidden_size: int = 64,
    num_layers: int = 1,
    dropout: float = 0.0,
    learning_rate: float = 1e-3,
    weight_decay: float = 0.0,
    batch_size: int = 64,
    eval_batch_size: int = 64,
    epochs: int = 20,
    patience: int = 5,
    random_state: int = 42,
    device: str | None = None,
    deterministic: bool = True,
    class_weight=None,
    num_workers: int = 0,
    optimizer_name: str = "adam",
    grad_clip_norm: float | None = None,
    verbose: bool = False,
):
    """
    Ejecuta GRU para un bundle seq2one.
    Evalúa SOLO sobre VALID.

    - Usa TRAIN para fit
    - Usa VALID para early stopping
    - Predice SOLO en VALID
    - Soporta labels arbitrarias (ej. [-1, 0, 1]) mediante codificación interna
    """

    # =========================
    # 1. SEEDS Y DEVICE
    # =========================
    torch.manual_seed(random_state)
    np.random.seed(random_state)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(random_state)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    use_pin_memory = device == "cuda"

    # =========================
    # 2. EXTRAER DATA
    # =========================
    X_train = bundle["train"]["X"]
    y_train = bundle["train"]["y"]

    X_valid = bundle["valid"]["X"]
    y_valid = bundle["valid"]["y"]

    target = bundle.get("target")
    horizon = bundle.get("horizon")
    window_size = bundle.get("window_size")

    # =========================
    # 3. VALIDAR SHAPES
    # =========================
    if X_train.ndim != 3 or X_valid.ndim != 3:
        raise ValueError(
            "GRU requiere tensores 3D: (n_samples, seq_len, n_features). "
            f"Recibido train={X_train.shape}, valid={X_valid.shape}"
        )

    seq_len_train, n_features_train = X_train.shape[1], X_train.shape[2]
    seq_len_valid, n_features_valid = X_valid.shape[1], X_valid.shape[2]

    if not (
        seq_len_train == seq_len_valid
        and n_features_train == n_features_valid
    ):
        raise ValueError(
            "Inconsistencia entre shapes de train/valid. "
            f"train={X_train.shape}, valid={X_valid.shape}"
        )

    n_features = n_features_train

    # =========================
    # 4. CODIFICAR LABELS
    # =========================
    classes_ = np.sort(np.unique(y_train))

    unknown_valid = set(np.unique(y_valid)) - set(classes_)
    if unknown_valid:
        raise ValueError(
            f"VALID contiene clases no vistas en TRAIN: {sorted(unknown_valid)}"
        )

    class_to_idx = {cls: idx for idx, cls in enumerate(classes_)}
    idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

    y_train_enc = np.array([class_to_idx[y] for y in y_train], dtype=np.int64)
    y_valid_enc = np.array([class_to_idx[y] for y in y_valid], dtype=np.int64)

    num_classes = len(classes_)

    # Validación opcional útil para este proyecto
    if not np.array_equal(classes_, [-1, 0, 1]):
        raise ValueError(f"Clases inesperadas en TRAIN: {classes_}")

    # =========================
    # 5. CLASS WEIGHTS
    # =========================
    criterion_weight = None
    weights_by_idx = None

    if class_weight is None:
        criterion_weight = None

    elif class_weight == "balanced":
        counts = np.bincount(y_train_enc, minlength=num_classes)
        total = counts.sum()

        weights_by_idx = {
            idx: total / (num_classes * count)
            for idx, count in enumerate(counts)
        }

        criterion_weight = torch.tensor(
            [weights_by_idx[idx] for idx in range(num_classes)],
            dtype=torch.float32,
            device=device,
        )

    elif isinstance(class_weight, dict):
        weights_by_idx = {
            class_to_idx[cls]: weight
            for cls, weight in class_weight.items()
            if cls in class_to_idx
        }

        criterion_weight = torch.tensor(
            [weights_by_idx.get(idx, 1.0) for idx in range(num_classes)],
            dtype=torch.float32,
            device=device,
        )

    else:
        raise ValueError("class_weight debe ser None, 'balanced' o dict")

    # =========================
    # 6. TENSORES EN CPU
    # =========================
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train_enc, dtype=torch.long)

    X_valid_t = torch.tensor(X_valid, dtype=torch.float32)
    y_valid_t = torch.tensor(y_valid_enc, dtype=torch.long)

    # =========================
    # 7. DATALOADERS
    # =========================
    train_ds = torch.utils.data.TensorDataset(X_train_t, y_train_t)
    valid_ds = torch.utils.data.TensorDataset(X_valid_t, y_valid_t)

    train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    valid_loader = torch.utils.data.DataLoader(
        valid_ds,
        batch_size=eval_batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    # =========================
    # 8. MODELO
    # =========================
    model = GRUClassifier(
        n_features=n_features,
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout,
        num_classes=num_classes,
    ).to(device)

    criterion = nn.CrossEntropyLoss(weight=criterion_weight)

    # =========================
    # 9. OPTIMIZER
    # =========================
    optimizer_name_norm = optimizer_name.lower()

    if optimizer_name_norm == "adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay,
        )
    elif optimizer_name_norm == "adamw":
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay,
        )
    else:
        raise ValueError(f"optimizer_name no soportado: {optimizer_name}")

    # =========================
    # 10. HELPERS
    # =========================
    def _move_batch(x):
        if device == "cuda":
            return x.to(device, non_blocking=True)
        return x.to(device)

    def compute_valid_loss():
        model.eval()
        valid_loss_sum = 0.0
        valid_count = 0

        with torch.no_grad():
            for xb, yb in valid_loader:
                xb = _move_batch(xb)
                yb = _move_batch(yb)

                logits = model(xb)
                loss = criterion(logits, yb)

                batch_n = xb.size(0)
                valid_loss_sum += loss.item() * batch_n
                valid_count += batch_n

        return valid_loss_sum / max(valid_count, 1)

    def predict_loader(loader):
        logits_all = []

        model.eval()
        with torch.no_grad():
            for xb, *_ in loader:
                xb = _move_batch(xb)
                logits = model(xb)
                logits_all.append(logits.cpu())

        logits_all = torch.cat(logits_all, dim=0)
        return logits_all

    # =========================
    # 11. TRAIN + EARLY STOPPING
    # =========================
    best_state = copy.deepcopy(model.state_dict())
    best_valid_loss = np.inf
    best_epoch = 0
    wait = 0
    history = []

    try:
        for epoch in range(1, epochs + 1):
            model.train()
            train_loss_sum = 0.0
            train_count = 0

            for xb, yb in train_loader:
                xb = _move_batch(xb)
                yb = _move_batch(yb)

                optimizer.zero_grad(set_to_none=True)
                logits = model(xb)
                loss = criterion(logits, yb)

                loss.backward()

                if grad_clip_norm is not None:
                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        grad_clip_norm,
                    )

                optimizer.step()

                batch_n = xb.size(0)
                train_loss_sum += loss.item() * batch_n
                train_count += batch_n

            train_loss = train_loss_sum / max(train_count, 1)
            valid_loss = compute_valid_loss()

            history.append(
                {
                    "epoch": epoch,
                    "train_loss": float(train_loss),
                    "valid_loss": float(valid_loss),
                }
            )

            if verbose:
                print(
                    f"[Epoch {epoch:03d}] "
                    f"train_loss={train_loss:.6f} | "
                    f"valid_loss={valid_loss:.6f}"
                )

            if valid_loss < best_valid_loss:
                best_valid_loss = valid_loss
                best_epoch = epoch
                best_state = copy.deepcopy(model.state_dict())
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    if verbose:
                        print(
                            f"[EARLY STOP] epoch={epoch} | "
                            f"best_epoch={best_epoch} | "
                            f"best_valid_loss={best_valid_loss:.6f}"
                        )
                    break

        model.load_state_dict(best_state)

        # =========================
        # 12. PREDICT (VALID)
        # =========================
        valid_logits = predict_loader(valid_loader)

        y_pred_valid_enc = valid_logits.argmax(dim=1).numpy()
        y_proba_valid = torch.softmax(valid_logits, dim=1).numpy()
        y_pred_valid = np.array([idx_to_class[int(y)] for y in y_pred_valid_enc])

        if verbose:
            print(
                f"[GRU] target={target} | horizon={horizon} | "
                f"window_size={window_size} | device={device} | "
                f"X_train={X_train.shape} | X_valid={X_valid.shape} | "
                f"best_epoch={best_epoch} | best_valid_loss={best_valid_loss:.6f}"
            )

        return {
            "model_name": "gru",
            "target": target,
            "horizon": horizon,
            "window_size": window_size,

            # hiperparámetros
            "hidden_size": hidden_size,
            "num_layers": num_layers,
            "dropout": dropout,
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "batch_size": batch_size,
            "eval_batch_size": eval_batch_size,
            "epochs": epochs,
            "patience": patience,
            "random_state": random_state,
            "deterministic": deterministic,
            "class_weight": str(class_weight),
            "num_workers": num_workers,
            "optimizer_name": optimizer_name_norm,
            "grad_clip_norm": grad_clip_norm,

            # estado / metadata
            "device": device,
            "model": model,
            "classes_": classes_.tolist(),
            "class_to_idx": class_to_idx,
            "idx_to_class": idx_to_class,
            "criterion_weight": (
                criterion_weight.detach().cpu().numpy().tolist()
                if criterion_weight is not None else None
            ),
            "weights_by_idx": weights_by_idx,
            "history": history,
            "best_valid_loss": float(best_valid_loss),
            "best_epoch": int(best_epoch),

            # outputs
            "y_valid": y_valid,
            "y_pred_valid": y_pred_valid,
            "y_proba_valid": y_proba_valid,
        }

    finally:
        if device == "cuda":
            torch.cuda.empty_cache()

## **10.2. Función de evaluación sobre uno o más bundles**

In [20]:
from typing import Any, Dict, List, Sequence, Union
import gc
import pandas as pd
import torch


def eval_gru_bundles(
    bundles: Union[Dict[str, Any], Sequence[Dict[str, Any]]],
    *,
    model_name: str = "gru",
    hidden_size: int = 64,
    num_layers: int = 1,
    dropout: float = 0.0,
    learning_rate: float = 1e-3,
    weight_decay: float = 0.0,
    batch_size: int = 64,
    eval_batch_size: int = 64,
    epochs: int = 20,
    patience: int = 5,
    random_state: int = 42,
    device: str | None = None,
    class_weight=None,
    num_workers: int = 0,
    optimizer_name: str = "adam",
    grad_clip_norm: float | None = None,
    prob_threshold_long: float = 0.40,
    prob_threshold_short: float = 0.40,
    verbose: bool = False,
) -> Dict[str, pd.DataFrame]:
    """
    Evalúa GRU para uno o varios bundles seq2one
    usando SOLO el split VALID.

    Retorna
    -------
    dict con:
    - "metrics": DataFrame de métricas de clasificación
    - "probabilities": DataFrame de métricas probabilísticas / decisión
    """

    # --------------------------------------------------
    # 1) Normalizar entrada
    # --------------------------------------------------
    if isinstance(bundles, dict):
        if "train" in bundles and "valid" in bundles:
            bundles_list: List[Dict[str, Any]] = [bundles]
        else:
            bundles_list = list(bundles.values())
    else:
        bundles_list = list(bundles)

    metrics_rows = []
    probabilities_rows = []

    optimizer_name_norm = optimizer_name.lower()

    # --------------------------------------------------
    # 2) Iterar por bundles
    # --------------------------------------------------
    for bundle in bundles_list:
        target = bundle.get("target", None)
        window_size = int(bundle["window_size"])
        horizon = int(bundle.get("horizon", -1))

        if verbose:
            print(
                f"  -> L{window_size} | "
                f"target={target} | "
                f"model={model_name} | "
                f"class_weight={class_weight}"
            )

        preds = None

        try:
            # ----------------------------------------------
            # 3) Entrenar + predecir SOLO VALID
            # ----------------------------------------------
            preds = run_gru_for_bundle_seq2one(
                bundle,
                hidden_size=hidden_size,
                num_layers=num_layers,
                dropout=dropout,
                learning_rate=learning_rate,
                weight_decay=weight_decay,
                batch_size=batch_size,
                eval_batch_size=eval_batch_size,
                epochs=epochs,
                patience=patience,
                random_state=random_state,
                device=device,
                class_weight=class_weight,
                num_workers=num_workers,
                optimizer_name=optimizer_name_norm,
                grad_clip_norm=grad_clip_norm,
                verbose=False,
            )

            y_true = preds["y_valid"]
            y_pred = preds["y_pred_valid"]
            y_proba = preds["y_proba_valid"]

            # ----------------------------------------------
            # 4) Métricas de clasificación
            # ----------------------------------------------
            metrics = compute_classification_metrics(
                y_true=y_true,
                y_pred=y_pred,
                model_name=model_name,
                split="valid",
                target=target,
                labels=[-1, 0, 1],
            )

            df_metrics_row = classification_metrics_to_df(
                metrics,
                model=model_name,
                split="valid",
                window_size=window_size,
                target=target,
                horizon=horizon,
            )

            if class_weight == "balanced":
                class_weight_mode = "balanced"
            elif class_weight is None:
                class_weight_mode = "none"
            else:
                class_weight_mode = "custom"

            df_metrics_row["class_weight_mode"] = class_weight_mode
            df_metrics_row["input_mode"] = "3d"
            df_metrics_row["hidden_size"] = hidden_size
            df_metrics_row["num_layers"] = num_layers
            df_metrics_row["dropout"] = dropout
            df_metrics_row["learning_rate"] = learning_rate
            df_metrics_row["weight_decay"] = weight_decay
            df_metrics_row["batch_size"] = batch_size
            df_metrics_row["eval_batch_size"] = eval_batch_size
            df_metrics_row["epochs"] = epochs
            df_metrics_row["patience"] = patience
            df_metrics_row["device"] = preds.get("device", device)
            df_metrics_row["best_epoch"] = preds.get("best_epoch")
            df_metrics_row["best_valid_loss"] = preds.get("best_valid_loss")
            df_metrics_row["optimizer_name"] = optimizer_name_norm
            df_metrics_row["grad_clip_norm"] = grad_clip_norm
            df_metrics_row["num_workers"] = num_workers
            df_metrics_row["random_state"] = random_state
            df_metrics_row["threshold_long"] = prob_threshold_long
            df_metrics_row["threshold_short"] = prob_threshold_short

            metrics_rows.append(df_metrics_row)

            # ----------------------------------------------
            # 5) Outputs probabilísticos
            # ----------------------------------------------
            class_labels = preds["classes_"]

            proba_df = compute_probabilistic_outputs(
                y_proba=y_proba,
                class_labels=class_labels,
                y_true=y_true,
            )

            decision_df = apply_decision_rule(
                proba_df,
                long_class=1,
                short_class=-1,
                long_threshold=prob_threshold_long,
                short_threshold=prob_threshold_short,
            )

            overlap_cols = [c for c in decision_df.columns if c in proba_df.columns]
            if overlap_cols:
                decision_df = decision_df.drop(columns=overlap_cols)

            df_prob = pd.concat(
                [proba_df.reset_index(drop=True), decision_df.reset_index(drop=True)],
                axis=1,
            )

            df_prob["model"] = model_name
            df_prob["split"] = "valid"
            df_prob["window_size"] = window_size
            df_prob["target"] = target
            df_prob["horizon"] = horizon
            df_prob["class_weight_mode"] = class_weight_mode
            df_prob["input_mode"] = "3d"
            df_prob["hidden_size"] = hidden_size
            df_prob["num_layers"] = num_layers
            df_prob["dropout"] = dropout
            df_prob["learning_rate"] = learning_rate
            df_prob["weight_decay"] = weight_decay
            df_prob["batch_size"] = batch_size
            df_prob["eval_batch_size"] = eval_batch_size
            df_prob["epochs"] = epochs
            df_prob["patience"] = patience
            df_prob["optimizer_name"] = optimizer_name_norm
            df_prob["grad_clip_norm"] = grad_clip_norm
            df_prob["num_workers"] = num_workers
            df_prob["device"] = preds.get("device", device)
            df_prob["best_epoch"] = preds.get("best_epoch")
            df_prob["best_valid_loss"] = preds.get("best_valid_loss")
            df_prob["random_state"] = random_state
            df_prob["threshold_long"] = prob_threshold_long
            df_prob["threshold_short"] = prob_threshold_short

            # sample_id para incremental
            df_prob = df_prob.reset_index(drop=True)
            df_prob["sample_id"] = df_prob.index.astype(int)

            probabilities_rows.append(df_prob)

        finally:
            # ----------------------------------------------
            # 6) Liberación explícita de memoria
            # ----------------------------------------------
            if preds is not None:
                del preds
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # --------------------------------------------------
    # 7) Consolidar salida
    # --------------------------------------------------
    df_metrics_all = pd.concat(metrics_rows, ignore_index=True)
    df_probabilities_all = pd.concat(probabilities_rows, ignore_index=True)

    return {
        "metrics": df_metrics_all,
        "probabilities": df_probabilities_all,
    }

## **10.3. Función orquestadora por `window_size`**

In [24]:
import gc
import pandas as pd
import torch


def run_gru(
    window_size: int,
    *,
    targets: list[str] = TARGETS,
    verbose: bool = True,
    model_name: str = "gru",
    hidden_size: int = 32,
    num_layers: int = 1,
    dropout: float = 0.0,
    learning_rate: float = 1e-3,
    weight_decay: float = 0.0,
    batch_size: int = 32,
    eval_batch_size: int = 32,
    epochs: int = 20,
    patience: int = 5,
    random_state: int = 42,
    device: str | None = None,
    class_weight=None,
    num_workers: int = 0,
    optimizer_name: str = "adam",
    grad_clip_norm: float | None = None,
    prob_threshold_long: float = 0.40,
    prob_threshold_short: float = 0.40,
) -> dict[str, pd.DataFrame]:
    """
    Ejecuta GRU para una sola window_size sobre los targets T2 indicados.

    Evalúa SOLO sobre VALID.

    Retorna
    -------
    dict con:
    - "metrics": DataFrame consolidado de métricas de clasificación
    - "probabilities": DataFrame consolidado de métricas probabilísticas / decisión
    """

    size = int(window_size)

    bundles = None
    results = None

    optimizer_name_norm = optimizer_name.lower()
    model_name_effective = model_name

    try:
        # --------------------------------------------------
        # 1) Encabezado
        # --------------------------------------------------
        if verbose:
            print("\n" + "=" * 80)
            print(f"GRU | T2 SEQ2ONE | WINDOW_SIZE=L{size}")
            print("=" * 80)
            print(f"targets          = {targets}")
            print(f"class_weight     = {class_weight}")
            print(f"hidden_size      = {hidden_size}")
            print(f"num_layers       = {num_layers}")
            print(f"dropout          = {dropout}")
            print(f"learning_rate    = {learning_rate}")
            print(f"weight_decay     = {weight_decay}")
            print(f"batch_size       = {batch_size}")
            print(f"eval_batch_size  = {eval_batch_size}")
            print(f"epochs           = {epochs}")
            print(f"patience         = {patience}")
            print(f"optimizer_name   = {optimizer_name_norm}")
            print(f"grad_clip_norm   = {grad_clip_norm}")
            print(f"device           = {device}")
            print(f"thr_long         = {prob_threshold_long}")
            print(f"thr_short        = {prob_threshold_short}")

        # --------------------------------------------------
        # 2) Construcción de bundles
        # --------------------------------------------------
        if verbose:
            print(f"\n[BUILD] L{size} | n_targets={len(targets)}")

        bundles = create_bundles(
            window_size=size,
            targets=targets,
            windows_paths=WINDOWS_PATHS,
            scaler_path=SCALER_T2_PATH,
        )

        # --------------------------------------------------
        # 3) Evaluación SOLO VALID
        # --------------------------------------------------
        if verbose:
            print(
                f"\n[EVAL] L{size} | split=valid | "
                f"model={model_name_effective} | class_weight={class_weight}"
            )

        results = eval_gru_bundles(
            bundles=bundles,
            model_name=model_name_effective,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            batch_size=batch_size,
            eval_batch_size=eval_batch_size,
            epochs=epochs,
            patience=patience,
            random_state=random_state,
            device=device,
            class_weight=class_weight,
            num_workers=num_workers,
            optimizer_name=optimizer_name_norm,
            grad_clip_norm=grad_clip_norm,
            prob_threshold_long=prob_threshold_long,
            prob_threshold_short=prob_threshold_short,
            verbose=verbose,
        )

        df_metrics = (
            results["metrics"]
            .sort_values(["window_size", "target", "split", "horizon", "model"])
            .reset_index(drop=True)
        )

        df_probabilities = (
            results["probabilities"]
            .sort_values(["window_size", "target", "split", "horizon", "model"])
            .reset_index(drop=True)
        )

        # --------------------------------------------------
        # 4) Resumen final
        # --------------------------------------------------
        if verbose:
            print(
                f"\n[DONE] L{size} | "
                f"metrics_rows={len(df_metrics)} | "
                f"probabilities_rows={len(df_probabilities)}"
            )

            print("\n[METRICS]")
            cols_metrics = [
                c for c in [
                    "window_size",
                    "split",
                    "target",
                    "model",
                    "horizon",
                    "class_weight_mode",
                    "balanced_accuracy",
                    "f1_macro",
                    "best_epoch",
                    "best_valid_loss",
                ] if c in df_metrics.columns
            ]
            if cols_metrics:
                print(df_metrics[cols_metrics].to_string(index=False))

            print("\n[PROBABILITIES - unique rows]")
            cols_probs = [
                c for c in [
                    "window_size",
                    "split",
                    "target",
                    "model",
                    "horizon",
                    "class_weight_mode",
                    "threshold_long",
                    "threshold_short",
                ] if c in df_probabilities.columns
            ]
            if cols_probs:
                print(
                    df_probabilities[cols_probs]
                    .drop_duplicates()
                    .to_string(index=False)
                )

        return {
            "metrics": df_metrics,
            "probabilities": df_probabilities,
        }

    finally:
        # --------------------------------------------------
        # 5) Liberación de memoria
        # --------------------------------------------------
        del bundles, results
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## **10.4. Función incremental de tuneo**

In [25]:
from itertools import product
import pandas as pd


def run_gru_grid_incremental(
    window_size: int,
    *,
    targets: list[str],
    hidden_size_values: list[int],
    num_layers_values: list[int],
    learning_rate_values: list[float],
    dropout_values: list[float],
    batch_size_values: list[int],
    grad_clip_norm_values: list[float | None],
    threshold_long_values: list[float],
    threshold_short_values: list[float],
    model_name: str = "gru",
    weight_decay: float = 0.0,
    eval_batch_size: int | None = None,
    epochs: int = 20,
    patience: int = 5,
    random_state: int = 42,
    device: str | None = None,
    class_weight=None,
    num_workers: int = 0,
    optimizer_name: str = "adam",
    split: str = "valid",
    verbose: bool = True,
) -> dict[str, pd.DataFrame]:

    # --------------------------------------------------
    # 0) Helpers
    # --------------------------------------------------
    def safe_eq(df, col, value):
        if col in df.columns:
            return df[col] == value
        return pd.Series(False, index=df.index)

    optimizer_name_norm = optimizer_name.lower()

    if eval_batch_size is None:
        # por consistencia, usar mismo batch_size que train
        # se redefine dentro del loop para cada combinación
        pass

    if class_weight == "balanced":
        class_weight_mode = "balanced"
    elif class_weight is None:
        class_weight_mode = "none"
    else:
        class_weight_mode = "custom"

    # --------------------------------------------------
    # 1) Cargar persistencia previa
    # --------------------------------------------------
    df_metrics_existing = load_classification_metrics_if_exists(
        model_name=model_name,
        split=split,
    )

    df_prob_existing = load_classification_probabilities_if_exists(
        model_name=model_name,
        split=split,
    )

    # --------------------------------------------------
    # 2) Asegurar columnas requeridas
    # --------------------------------------------------
    required_metric_cols = [
        "model", "split", "window_size", "target",
        "hidden_size", "num_layers", "dropout",
        "learning_rate", "weight_decay",
        "batch_size", "eval_batch_size",
        "epochs", "patience",
        "input_mode", "class_weight_mode",
        "optimizer_name", "grad_clip_norm",
        "num_workers", "random_state",
        "threshold_long", "threshold_short",
    ]

    required_prob_cols = required_metric_cols + ["sample_id"]

    for col in required_metric_cols:
        if col not in df_metrics_existing.columns:
            df_metrics_existing[col] = None

    for col in required_prob_cols:
        if col not in df_prob_existing.columns:
            df_prob_existing[col] = None

    # --------------------------------------------------
    # 3) Definir grilla
    # --------------------------------------------------
    grid = list(product(
        hidden_size_values,
        num_layers_values,
        learning_rate_values,
        dropout_values,
        batch_size_values,
        grad_clip_norm_values,
        threshold_long_values,
        threshold_short_values,
    ))

    if verbose:
        print("\n" + "=" * 100)
        print(f"GRU GRID INCREMENTAL | L={window_size}")
        print("=" * 100)
        print(f"targets                = {targets}")
        print(f"n_combinations         = {len(grid)}")
        print(f"hidden_size_values     = {hidden_size_values}")
        print(f"num_layers_values      = {num_layers_values}")
        print(f"learning_rate_values   = {learning_rate_values}")
        print(f"dropout_values         = {dropout_values}")
        print(f"batch_size_values      = {batch_size_values}")
        print(f"grad_clip_norm_values  = {grad_clip_norm_values}")
        print(f"threshold_long_values  = {threshold_long_values}")
        print(f"threshold_short_values = {threshold_short_values}")
        print(f"class_weight_mode      = {class_weight_mode}")
        print(f"optimizer_name         = {optimizer_name_norm}")

    # --------------------------------------------------
    # 4) Loop principal
    # --------------------------------------------------
    for i, (
        hidden_size,
        num_layers,
        learning_rate,
        dropout,
        batch_size,
        grad_clip_norm,
        thr_long,
        thr_short,
    ) in enumerate(grid, start=1):

        eval_bs_current = eval_batch_size if eval_batch_size is not None else batch_size

        if verbose:
            print("\n" + "-" * 100)
            print(
                f"[{i}/{len(grid)}] "
                f"hidden_size={hidden_size} | "
                f"num_layers={num_layers} | "
                f"learning_rate={learning_rate} | "
                f"dropout={dropout} | "
                f"batch_size={batch_size} | "
                f"eval_batch_size={eval_bs_current} | "
                f"grad_clip_norm={grad_clip_norm} | "
                f"thr_long={thr_long} | "
                f"thr_short={thr_short}"
            )

        # ----------------------------------------------
        # 4.1) Verificar si el experimento ya existe
        # ----------------------------------------------
        mask = (
            safe_eq(df_metrics_existing, "model", model_name) &
            safe_eq(df_metrics_existing, "split", split) &
            safe_eq(df_metrics_existing, "window_size", window_size) &
            safe_eq(df_metrics_existing, "hidden_size", hidden_size) &
            safe_eq(df_metrics_existing, "num_layers", num_layers) &
            safe_eq(df_metrics_existing, "dropout", dropout) &
            safe_eq(df_metrics_existing, "learning_rate", learning_rate) &
            safe_eq(df_metrics_existing, "weight_decay", weight_decay) &
            safe_eq(df_metrics_existing, "batch_size", batch_size) &
            safe_eq(df_metrics_existing, "eval_batch_size", eval_bs_current) &
            safe_eq(df_metrics_existing, "epochs", epochs) &
            safe_eq(df_metrics_existing, "patience", patience) &
            safe_eq(df_metrics_existing, "input_mode", "3d") &
            safe_eq(df_metrics_existing, "class_weight_mode", class_weight_mode) &
            safe_eq(df_metrics_existing, "optimizer_name", optimizer_name_norm) &
            safe_eq(df_metrics_existing, "grad_clip_norm", grad_clip_norm) &
            safe_eq(df_metrics_existing, "num_workers", num_workers) &
            safe_eq(df_metrics_existing, "random_state", random_state) &
            safe_eq(df_metrics_existing, "threshold_long", thr_long) &
            safe_eq(df_metrics_existing, "threshold_short", thr_short)
        )

        existing_targets = set(df_metrics_existing.loc[mask, "target"].dropna().unique())
        already_exists = set(targets).issubset(existing_targets)

        if already_exists:
            if verbose:
                print("✔ Ya existe -> skip")
            continue

        # ----------------------------------------------
        # 4.2) Ejecutar modelo
        # ----------------------------------------------
        results = run_gru(
            window_size=window_size,
            targets=targets,
            verbose=verbose,
            model_name=model_name,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            batch_size=batch_size,
            eval_batch_size=eval_bs_current,
            epochs=epochs,
            patience=patience,
            random_state=random_state,
            device=device,
            class_weight=class_weight,
            num_workers=num_workers,
            optimizer_name=optimizer_name_norm,
            grad_clip_norm=grad_clip_norm,
            prob_threshold_long=thr_long,
            prob_threshold_short=thr_short,
        )

        df_metrics_new = results["metrics"].copy()
        df_prob_new = results["probabilities"].copy()

        # ----------------------------------------------
        # 4.3) Completar metadata
        # ----------------------------------------------
        df_metrics_new["threshold_long"] = thr_long
        df_metrics_new["threshold_short"] = thr_short
        df_metrics_new["random_state"] = random_state

        df_prob_new["threshold_long"] = thr_long
        df_prob_new["threshold_short"] = thr_short
        df_prob_new["random_state"] = random_state

        df_prob_new = df_prob_new.reset_index(drop=True)
        if "sample_id" not in df_prob_new.columns:
            df_prob_new["sample_id"] = df_prob_new.index.astype(int)

        for col in required_metric_cols:
            if col not in df_metrics_new.columns:
                df_metrics_new[col] = None

        for col in required_prob_cols:
            if col not in df_prob_new.columns:
                df_prob_new[col] = None

        # ----------------------------------------------
        # 4.4) Append
        # ----------------------------------------------
        df_metrics_existing = pd.concat(
            [df_metrics_existing, df_metrics_new],
            ignore_index=True
        )

        df_prob_existing = pd.concat(
            [df_prob_existing, df_prob_new],
            ignore_index=True
        )

        # ----------------------------------------------
        # 4.5) Deduplicación correcta
        # ----------------------------------------------
        metric_key_cols = [
            "model", "split", "window_size", "target",
            "hidden_size", "num_layers", "dropout",
            "learning_rate", "weight_decay",
            "batch_size", "eval_batch_size",
            "epochs", "patience",
            "input_mode", "class_weight_mode",
            "optimizer_name", "grad_clip_norm",
            "num_workers", "random_state",
            "threshold_long", "threshold_short",
        ]

        prob_key_cols = metric_key_cols + ["sample_id"]

        df_metrics_existing = (
            df_metrics_existing
            .drop_duplicates(subset=metric_key_cols, keep="last")
            .reset_index(drop=True)
        )

        df_prob_existing = (
            df_prob_existing
            .drop_duplicates(subset=prob_key_cols, keep="last")
            .reset_index(drop=True)
        )

        # ----------------------------------------------
        # 4.6) Guardar
        # ----------------------------------------------
        save_classification_metrics(
            df_metrics_existing,
            model_name=model_name,
            split=split,
        )

        save_classification_probabilities(
            df_prob_existing,
            model_name=model_name,
            split=split,
        )

        if verbose:
            print(
                f"💾 Guardado OK | metrics={len(df_metrics_existing)} | "
                f"prob={len(df_prob_existing)}"
            )

    # --------------------------------------------------
    # 5) Retorno final
    # --------------------------------------------------
    return {
        "metrics": df_metrics_existing,
        "probabilities": df_prob_existing,
    }

# **11. Tuneo grueso**

In [26]:
results_gru_coarse = run_gru_grid_incremental(
    window_size=30,
    targets=["t2_p40_h30", "t2_p50_h30"],
    hidden_size_values=[64, 128, 256],
    num_layers_values=[1, 2],
    learning_rate_values=[1e-4, 5e-4, 1e-3],
    dropout_values=[0.0],
    batch_size_values=[2048],
    grad_clip_norm_values=[1.0],
    threshold_long_values=[0.40],
    threshold_short_values=[0.40],
    class_weight="balanced",
    epochs=20,
    patience=5,
    optimizer_name="adam",
    verbose=True,
)

2026-04-23 16:28:01,375 | INFO | No existen métricas previas para model=gru | split=valid
2026-04-23 16:28:01,377 | INFO | No existen probabilidades previas para model=gru | split=valid
2026-04-23 16:28:01,462 | INFO | Loaded: windows_t2_p40_h30_train.npz
2026-04-23 16:28:01,463 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-23 16:28:01,480 | INFO | Loaded: windows_t2_p40_h30_valid.npz
2026-04-23 16:28:01,480 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-23 16:28:01,497 | INFO | Loaded: windows_t2_p40_h30_test.npz
2026-04-23 16:28:01,498 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-23 16:28:01,501 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-23 16:28:01,501 | INFO | Bundle cargado | target=t2_p40_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)
2026-04-23 16:28:01,575 | INFO | Loaded: windows_t2_p50_h30_train.npz
2026-04-23 16:28:01,576 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)



GRU GRID INCREMENTAL | L=30
targets                = ['t2_p40_h30', 't2_p50_h30']
n_combinations         = 18
hidden_size_values     = [64, 128, 256]
num_layers_values      = [1, 2]
learning_rate_values   = [0.0001, 0.0005, 0.001]
dropout_values         = [0.0]
batch_size_values      = [2048]
grad_clip_norm_values  = [1.0]
threshold_long_values  = [0.4]
threshold_short_values = [0.4]
class_weight_mode      = balanced
optimizer_name         = adam

----------------------------------------------------------------------------------------------------
[1/18] hidden_size=64 | num_layers=1 | learning_rate=0.0001 | dropout=0.0 | batch_size=2048 | eval_batch_size=2048 | grad_clip_norm=1.0 | thr_long=0.4 | thr_short=0.4

GRU | T2 SEQ2ONE | WINDOW_SIZE=L30
targets          = ['t2_p40_h30', 't2_p50_h30']
class_weight     = balanced
hidden_size      = 64
num_layers       = 1
dropout          = 0.0
learning_rate    = 0.0001
weight_decay     = 0.0
batch_size       = 2048
eval_batch_size  = 2048
epoc

2026-04-23 16:28:01,593 | INFO | Loaded: windows_t2_p50_h30_valid.npz
2026-04-23 16:28:01,594 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-23 16:28:01,611 | INFO | Loaded: windows_t2_p50_h30_test.npz
2026-04-23 16:28:01,612 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-23 16:28:01,615 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-23 16:28:01,615 | INFO | Bundle cargado | target=t2_p50_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)



WINDOW_SIZE: 30

TARGET: t2_p40_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p50_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

[EVAL] L30 | split=valid | model=gru | class_weight=balanced
  -> L30 | target=t2_p40_h30 | model=gru | class_weight=balanced
  -> L30 | target=t2_p50_h30 | model=gru | class_weight=balanced

[DONE] L30 | metrics_rows=2 | probabilities_rows=13764

[METRICS]
 window_size split     target model  horizon class_weight_mode  balanced_accuracy  f1_macro  best_epoch  best_valid_loss
          30 valid t2_p40_h30   gru       30          balanced           0.408994  0.347568          10         1.087213
          30 valid t2_p50_h30   gru       30          balanced           0.408281  0.370536          10         1.087448

[PROBABILITIES - unique rows]
 window_size split     target model  horizon class_weight_mo

2026-04-23 16:28:14,352 | INFO | Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics_tuning/classification_metrics/classification_metrics_gru_valid.parquet
2026-04-23 16:28:14,410 | INFO | Probabilidades guardadas en: /content/drive/MyDrive/neural_profit/metrics_tuning/classification_probabilities/classification_probabilities_gru_valid.parquet
2026-04-23 16:28:14,486 | INFO | Loaded: windows_t2_p40_h30_train.npz
2026-04-23 16:28:14,486 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-23 16:28:14,503 | INFO | Loaded: windows_t2_p40_h30_valid.npz
2026-04-23 16:28:14,504 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-23 16:28:14,521 | INFO | Loaded: windows_t2_p40_h30_test.npz
2026-04-23 16:28:14,521 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-23 16:28:14,524 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-23 16:28:14,524 | INFO | Bundle cargado | target=t2_p40_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | tes

...

💾 Guardado OK | metrics=34 | prob=233988

----------------------------------------------------------------------------------------------------
[18/18] hidden_size=256 | num_layers=2 | learning_rate=0.001 | dropout=0.0 | batch_size=2048 | eval_batch_size=2048 | grad_clip_norm=1.0 | thr_long=0.4 | thr_short=0.4

GRU | T2 SEQ2ONE | WINDOW_SIZE=L30
targets          = ['t2_p40_h30', 't2_p50_h30']
class_weight     = balanced
hidden_size      = 256
num_layers       = 2
dropout          = 0.0
learning_rate    = 0.001
weight_decay     = 0.0
batch_size       = 2048
eval_batch_size  = 2048
epochs           = 20
patience         = 5
optimizer_name   = adam
grad_clip_norm   = 1.0
device           = None
thr_long         = 0.4
thr_short        = 0.4

[BUILD] L30 | n_targets=2


2026-04-23 16:31:25,703 | INFO | Loaded: windows_t2_p50_h30_test.npz
2026-04-23 16:31:25,704 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-23 16:31:25,707 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-23 16:31:25,707 | INFO | Bundle cargado | target=t2_p50_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)



WINDOW_SIZE: 30

TARGET: t2_p40_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p50_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

[EVAL] L30 | split=valid | model=gru | class_weight=balanced
  -> L30 | target=t2_p40_h30 | model=gru | class_weight=balanced
  -> L30 | target=t2_p50_h30 | model=gru | class_weight=balanced

[DONE] L30 | metrics_rows=2 | probabilities_rows=13764

[METRICS]
 window_size split     target model  horizon class_weight_mode  balanced_accuracy  f1_macro  best_epoch  best_valid_loss
          30 valid t2_p40_h30   gru       30          balanced           0.398097  0.378543           3         1.078617
          30 valid t2_p50_h30   gru       30          balanced           0.384949  0.371254           3         1.085100

[PROBABILITIES - unique rows]
 window_size split     target model  horizon class_weight_mo

2026-04-23 16:31:38,915 | INFO | Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics_tuning/classification_metrics/classification_metrics_gru_valid.parquet
2026-04-23 16:31:39,499 | INFO | Probabilidades guardadas en: /content/drive/MyDrive/neural_profit/metrics_tuning/classification_probabilities/classification_probabilities_gru_valid.parquet


💾 Guardado OK | metrics=36 | prob=247752


## **11.1. Análisis de tuneo grueso**

In [27]:
df_gru_coarse = results_gru_coarse["metrics"].copy()

# =========================================
# Resumen global por configuración
# =========================================
summary_gru = (
    df_gru_coarse
    .groupby(
        ["hidden_size", "num_layers", "learning_rate"],
        as_index=False
    )
    .agg(
        n_targets=("target", "nunique"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_std=("balanced_accuracy", "std"),
        f1_macro_mean=("f1_macro", "mean"),
        f1_macro_std=("f1_macro", "std"),
        accuracy_mean=("accuracy", "mean"),
    )
    .sort_values(
        ["balanced_accuracy_mean", "f1_macro_mean"],
        ascending=False
    )
    .reset_index(drop=True)
)

# =========================================
# Mejor configuración por target
# =========================================
best_gru_by_target = (
    df_gru_coarse
    .sort_values(
        ["target", "balanced_accuracy", "f1_macro"],
        ascending=[True, False, False]
    )
    .groupby("target", as_index=False)
    .first()[[
        "target",
        "hidden_size",
        "num_layers",
        "learning_rate",
        "balanced_accuracy",
        "f1_macro",
    ]]
)

# =========================================
# Mejor configuración global
# =========================================
best_gru_global = summary_gru.iloc[0][
    ["hidden_size", "num_layers", "learning_rate"]
].to_dict()

print("Resumen global por configuración GRU")
display(summary_gru)

print("Mejor configuración por target")
display(best_gru_by_target)

print("Mejor configuración global:")
print(best_gru_global)

Resumen global por configuración GRU


,hidden_size,num_layers,learning_rate,n_targets,balanced_accuracy_mean,balanced_accuracy_std,f1_macro_mean,f1_macro_std,accuracy_mean
0,128,2,0.0005,2,0.417345,0.004847,0.388284,0.011589,0.405623
1,128,1,0.0005,2,0.414957,0.001230,0.373755,0.020915,0.401555
2,256,1,0.0005,2,0.414275,0.001763,0.390369,0.008654,0.404897
3,128,1,0.0001,2,0.413745,0.000197,0.374423,0.016782,0.401337
4,128,2,0.0010,2,0.412285,0.000440,0.390434,0.016703,0.403153
5,64,2,0.0005,2,0.411791,0.005170,0.386085,0.011945,0.403662
6,64,2,0.0001,2,0.409817,0.003823,0.368986,0.000383,0.398431
7,64,1,0.0010,2,0.409078,0.006816,0.387753,0.006577,0.400610
8,256,2,0.0001,2,0.409028,0.007879,0.381275,0.000542,0.401482
9,128,1,0.0010,2,0.408738,0.010271,0.383748,0.001808,0.400320


Mejor configuración por target


,target,hidden_size,num_layers,learning_rate,balanced_accuracy,f1_macro
0,t2_p40_h30,128,2,0.0005,0.420773,0.380090
1,t2_p50_h30,256,1,0.0010,0.417474,0.407541


Mejor configuración global:
{'hidden_size': 128, 'num_layers': 2, 'learning_rate': 0.0005}


Gus, el resultado es bastante claro y consistente. Te lo dejo redactado en formato limpio para la notebook.

---

Lectura principal

La mejor configuración global obtenida en el tuneo grueso es:

* hidden_size = 128
* num_layers = 2
* learning_rate = 0.0005

Con desempeño promedio:

* balanced_accuracy ≈ 0.417
* f1_macro ≈ 0.388

Esta configuración se posiciona como la mejor en términos agregados entre ambos targets.

---

Patrón dominante

Se observa un comportamiento bastante estable:

* hidden_size = 128 aparece sistemáticamente en las mejores configuraciones
* learning_rate = 0.0005 domina claramente
* modelos con 2 capas (num_layers = 2) tienden a mejorar levemente el desempeño

Interpretación:

* el modelo requiere una capacidad intermedia (no 64, no necesariamente 256)
* un learning_rate moderado permite mejor convergencia
* aumentar la profundidad ayuda, pero con impacto acotado

---

Relación entre capacidad y estabilidad

Comparando configuraciones:

* hidden_size = 128 muestra menor varianza y mayor estabilidad
* hidden_size = 256 no mejora consistentemente y en algunos casos degrada
* hidden_size = 64 queda por debajo en general

Esto indica que:

* el modelo no necesita alta capacidad
* aumentar demasiado la complejidad no aporta mejora

---

Análisis por target

Los mejores resultados por target son:

* t2_p40_h30:

  * 128, 2, 0.0005
* t2_p50_h30:

  * 256, 1, 0.001

Esto muestra una ligera diferencia entre targets, pero:

* la configuración global (128, 2, 0.0005) es más estable
* la alternativa (256, 1, 0.001) es más específica

---

Selección para tuneo fino

Se recomienda trabajar con al menos dos configuraciones:

Configuración principal:

* hidden_size = 128
* num_layers = 2
* learning_rate = 0.0005

Configuración alternativa:

* hidden_size = 256
* num_layers = 1
* learning_rate = 0.001

Opcional (más conservadora):

* hidden_size = 128
* num_layers = 1
* learning_rate = 0.0005

---

Conclusión operativa

El modelo GRU presenta mejor desempeño cuando:

* se utiliza una capacidad intermedia
* se emplea un learning_rate moderado
* se permite una ligera profundidad adicional

El espacio óptimo queda bien definido, permitiendo enfocar el tuneo fino en:

* regularización (dropout)
* estabilidad de entrenamiento (batch_size, grad_clip_norm)
* thresholds de decisión

No es necesario explorar configuraciones más grandes o learning rates más agresivos.


# **12. Tuneo fino**

In [ ]:
results_gru_fine_1 = run_gru_grid_incremental(
    window_size=30,
    targets=["t2_p40_h30", "t2_p50_h30"],
    hidden_size_values=[128],
    num_layers_values=[2],
    learning_rate_values=[5e-4],
    dropout_values=[0.0, 0.1, 0.2, 0.3],
    batch_size_values=[1024, 2048, 4096],
    grad_clip_norm_values=[0.5, 1.0, 2.0],
    threshold_long_values=[0.40, 0.45],
    threshold_short_values=[0.40, 0.45],
    class_weight="balanced",
    epochs=20,
    patience=5,
    optimizer_name="adam",
    verbose=True,
)

results_gru_fine_2 = run_gru_grid_incremental(
    window_size=30,
    targets=["t2_p40_h30", "t2_p50_h30"],
    hidden_size_values=[256],
    num_layers_values=[1],
    learning_rate_values=[1e-3],
    dropout_values=[0.0, 0.1, 0.2, 0.3],
    batch_size_values=[1024, 2048, 4096],
    grad_clip_norm_values=[0.5, 1.0, 2.0],
    threshold_long_values=[0.40, 0.45],
    threshold_short_values=[0.40, 0.45],
    class_weight="balanced",
    epochs=20,
    patience=5,
    optimizer_name="adam",
    verbose=True,
)

results_gru_fine_3 = run_gru_grid_incremental(
    window_size=30,
    targets=["t2_p40_h30", "t2_p50_h30"],
    hidden_size_values=[128],
    num_layers_values=[1],
    learning_rate_values=[5e-4],
    dropout_values=[0.0, 0.1, 0.2, 0.3],
    batch_size_values=[1024, 2048, 4096],
    grad_clip_norm_values=[0.5, 1.0, 2.0],
    threshold_long_values=[0.40, 0.45],
    threshold_short_values=[0.40, 0.45],
    class_weight="balanced",
    epochs=20,
    patience=5,
    optimizer_name="adam",
    verbose=True,
)

2026-04-23 16:36:13,719 | INFO | Cargando métricas desde: /content/drive/MyDrive/neural_profit/metrics_tuning/classification_metrics/classification_metrics_gru_valid.parquet
2026-04-23 16:36:13,778 | INFO | Cargando probabilidades desde: /content/drive/MyDrive/neural_profit/metrics_tuning/classification_probabilities/classification_probabilities_gru_valid.parquet
2026-04-23 16:36:13,907 | INFO | Loaded: windows_t2_p40_h30_train.npz
2026-04-23 16:36:13,907 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-23 16:36:13,925 | INFO | Loaded: windows_t2_p40_h30_valid.npz
2026-04-23 16:36:13,925 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-23 16:36:13,942 | INFO | Loaded: windows_t2_p40_h30_test.npz
2026-04-23 16:36:13,942 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-23 16:36:13,945 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-23 16:36:13,946 | INFO | Bundle cargado | target=t2_p40_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) |


GRU GRID INCREMENTAL | L=30
targets                = ['t2_p40_h30', 't2_p50_h30']
n_combinations         = 144
hidden_size_values     = [128]
num_layers_values      = [2]
learning_rate_values   = [0.0005]
dropout_values         = [0.0, 0.1, 0.2, 0.3]
batch_size_values      = [1024, 2048, 4096]
grad_clip_norm_values  = [0.5, 1.0, 2.0]
threshold_long_values  = [0.4, 0.45]
threshold_short_values = [0.4, 0.45]
class_weight_mode      = balanced
optimizer_name         = adam

----------------------------------------------------------------------------------------------------
[1/144] hidden_size=128 | num_layers=2 | learning_rate=0.0005 | dropout=0.0 | batch_size=1024 | eval_batch_size=1024 | grad_clip_norm=0.5 | thr_long=0.4 | thr_short=0.4

GRU | T2 SEQ2ONE | WINDOW_SIZE=L30
targets          = ['t2_p40_h30', 't2_p50_h30']
class_weight     = balanced
hidden_size      = 128
num_layers       = 2
dropout          = 0.0
learning_rate    = 0.0005
weight_decay     = 0.0
batch_size       = 1024
ev

2026-04-23 16:36:14,047 | INFO | Loaded: windows_t2_p50_h30_test.npz
2026-04-23 16:36:14,048 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-23 16:36:14,051 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-23 16:36:14,052 | INFO | Bundle cargado | target=t2_p50_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)



WINDOW_SIZE: 30

TARGET: t2_p40_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p50_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

[EVAL] L30 | split=valid | model=gru | class_weight=balanced
  -> L30 | target=t2_p40_h30 | model=gru | class_weight=balanced
  -> L30 | target=t2_p50_h30 | model=gru | class_weight=balanced

[DONE] L30 | metrics_rows=2 | probabilities_rows=13764

[METRICS]
 window_size split     target model  horizon class_weight_mode  balanced_accuracy  f1_macro  best_epoch  best_valid_loss
          30 valid t2_p40_h30   gru       30          balanced           0.412523  0.385360           7         1.081819
          30 valid t2_p50_h30   gru       30          balanced           0.412331  0.400635           7         1.086069

[PROBABILITIES - unique rows]
 window_size split     target model  horizon class_weight_mo

2026-04-23 16:36:28,093 | INFO | Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics_tuning/classification_metrics/classification_metrics_gru_valid.parquet
2026-04-23 16:36:28,475 | INFO | Probabilidades guardadas en: /content/drive/MyDrive/neural_profit/metrics_tuning/classification_probabilities/classification_probabilities_gru_valid.parquet
2026-04-23 16:36:28,547 | INFO | Loaded: windows_t2_p40_h30_train.npz
2026-04-23 16:36:28,547 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-23 16:36:28,565 | INFO | Loaded: windows_t2_p40_h30_valid.npz
2026-04-23 16:36:28,566 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-23 16:36:28,583 | INFO | Loaded: windows_t2_p40_h30_test.npz
2026-04-23 16:36:28,584 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-23 16:36:28,587 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-23 16:36:28,588 | INFO | Bundle cargado | target=t2_p40_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | tes

💾 Guardado OK | metrics=38 | prob=261516

----------------------------------------------------------------------------------------------------
[2/144] hidden_size=128 | num_layers=2 | learning_rate=0.0005 | dropout=0.0 | batch_size=1024 | eval_batch_size=1024 | grad_clip_norm=0.5 | thr_long=0.4 | thr_short=0.45

GRU | T2 SEQ2ONE | WINDOW_SIZE=L30
targets          = ['t2_p40_h30', 't2_p50_h30']
class_weight     = balanced
hidden_size      = 128
num_layers       = 2
dropout          = 0.0
learning_rate    = 0.0005
weight_decay     = 0.0
batch_size       = 1024
eval_batch_size  = 1024
epochs           = 20
patience         = 5
optimizer_name   = adam
grad_clip_norm   = 0.5
device           = None
thr_long         = 0.4
thr_short        = 0.45

[BUILD] L30 | n_targets=2


2026-04-23 16:36:28,694 | INFO | Loaded: windows_t2_p50_h30_test.npz
2026-04-23 16:36:28,695 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-23 16:36:28,698 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-23 16:36:28,698 | INFO | Bundle cargado | target=t2_p50_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)



WINDOW_SIZE: 30

TARGET: t2_p40_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p50_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

[EVAL] L30 | split=valid | model=gru | class_weight=balanced
  -> L30 | target=t2_p40_h30 | model=gru | class_weight=balanced
  -> L30 | target=t2_p50_h30 | model=gru | class_weight=balanced


## **12.1. Análisis de tuneo fino**

Lecturas clave

1. Trade-off entre calidad y actividad
   Se observa el comportamiento esperado: thresholds más altos reducen la cantidad de operaciones pero incrementan la precisión, mientras que thresholds más bajos aumentan la actividad a costa de mayor ruido. La relación es consistente y no presenta anomalías.

2. Mejor precisión
   La combinación (0.45, 0.45) presenta la mayor precision_useful, pero con un nivel de actividad bajo. Esto corresponde a un perfil conservador, donde se prioriza calidad sobre volumen de señales.

3. Zona eficiente
   Las combinaciones más relevantes en términos de balance son:

* (0.40, 0.45): alta precisión con una reducción moderada de actividad
* (0.45, 0.40): comportamiento similar, con mayor volumen de señales

Ambas superan al baseline (0.40, 0.40), logrando una mejor relación entre calidad y cantidad de operaciones.

4. Baseline
   La configuración (0.40, 0.40) presenta mayor actividad pero menor precisión. Funciona como referencia, pero no es óptima en términos de eficiencia operativa.

Conclusión operativa

Se identifican tres perfiles de comportamiento:

* Conservador: (0.45, 0.45), máxima precisión y baja frecuencia
* Balanceado: (0.40, 0.45), alto nivel de precisión con buena reducción de ruido
* Agresivo: (0.35, 0.40), mayor actividad con menor calidad de señal

Recomendación

Se selecciona la siguiente configuración:

```python
threshold_long  = 0.40
threshold_short = 0.45
```

Esta combinación mantiene una precisión cercana al máximo, mejora el baseline y reduce significativamente el ruido sin afectar excesivamente la actividad.

Observación estructural

Se observa un patrón consistente donde las mejores combinaciones presentan threshold_short mayor que threshold_long. Esto sugiere que el modelo es más confiable en señales largas que en cortas, lo cual constituye una característica estructural del modelo.

Análisis adicional recomendado

* Evaluar la distribución de señales long y short
* Analizar desempeño por régimen de mercado
* Verificar estabilidad de señales a lo largo del tiempo
* Construir curvas de precision vs trade_rate para identificar puntos óptimos

Conclusión

El modelo responde adecuadamente al proceso de tuning y el espacio de thresholds está bien definido. Se ha identificado una zona operativa estable, lo que permite avanzar hacia validación en test o implementación de backtesting.


# **13. Selección final de hiperparámetros**

A partir del proceso de tuneo grueso y fino, se determina como configuración óptima del modelo:

```python
C = 0.01
```

Este valor presenta el mejor desempeño consistente en términos de balanced_accuracy y f1_macro, indicando la necesidad de una regularización fuerte para este problema.

En cuanto a los umbrales de decisión, el análisis operativo muestra un trade-off claro entre precisión y frecuencia de señales. A partir del ranking global, se seleccionan dos configuraciones representativas:

Configuración principal (balanceada):

```python
threshold_long  = 0.40
threshold_short = 0.45
```

* precision_useful alta (~0.447)
* reducción significativa del ruido respecto al baseline
* nivel de actividad moderado
* mejor relación señal / ruido

Configuración alternativa (conservadora):

```python
threshold_long  = 0.45
threshold_short = 0.45
```

* máxima precision_useful (~0.448)
* menor frecuencia de operaciones
* adecuada para escenarios donde se prioriza calidad sobre volumen

El baseline original:

```python
threshold_long  = 0.40
threshold_short = 0.40
```

queda superado por ambas configuraciones en términos de eficiencia operativa.

Observación relevante

Se identifica un patrón consistente donde las mejores configuraciones presentan:

```python
threshold_short > threshold_long
```

Esto sugiere que el modelo muestra mayor confiabilidad en señales largas que en señales cortas, constituyendo una característica estructural del modelo.

Conclusión

El modelo Logistic Regression queda definido por:

* regularización óptima: C = 0.01
* thresholds seleccionados en zona eficiente del espacio de decisión

Con esto se completa la etapa de selección de hiperparámetros para LR, quedando listo para su comparación con otros modelos en etapas posteriores.
